# Getting Pandas to work with MySQL
### Reading CSV files · Creating MySQL tables · Loading and querying data

---

**What this notebook covers - step by step:**

1. Install the required libraries
2. Connect to a MySQL database
3. Create a simple table and insert dummy data
4. Read real CSV files into pandas DataFrames
5. Write those DataFrames into MySQL tables
6. Read the data back from MySQL into pandas

**Datasets used:** `students.csv`, `teachers.csv`, `grades.csv`

---
## Step 1 - Install required libraries

`pymysql` lets Python talk to MySQL. `sqlalchemy` is the bridge that pandas uses to read and write database tables. Run this cell once, then restart the kernel if prompted.

In [1]:
# Install the two libraries needed for this tutorial
# pymysql  — MySQL driver (lets Python send SQL commands)
# sqlalchemy — used by pandas to connect to the database
%pip install pymysql sqlalchemy --quiet

Note: you may need to restart the kernel to use updated packages.


## Step 2 - Connect to MySQL

We build a connection string using standard MySQL credentials, then use SQLAlchemy to create an **engine** — think of this as the open door to your database. All future reads and writes go through this engine.

In [1]:
from sqlalchemy import create_engine, text
import pandas as pd
from getpass import getpass
# --- Your MySQL credentials ---
DB_USER     = 'root'           # MySQL username
DB_PASSWORD = getpass('Enter password')  # MySQL password
DB_HOST     = 'localhost'      # server address (localhost = same machine)
DB_PORT     = 3306             # default MySQL port
DB_NAME     = 'school_db'      # name of the database to use

# Build the connection string in the format SQLAlchemy expects
connection_string = f'mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'

# Create the engine — this opens the connection to MySQL
engine = create_engine(connection_string)

# Test the connection — if no error appears, we are connected
with engine.connect() as conn:
    print('Connected to MySQL successfully!')

Enter password ········


Connected to MySQL successfully!


> **Note:** If you see an `OperationalError`, check that MySQL is running and your credentials above are correct. The database `school_db` must already exist — create it in MySQL with `CREATE DATABASE school_db;` before running this notebook.

## Step 3 - Create a simple table and insert dummy data

Before working with real data, let's learn the basics: how to create a table in MySQL from Python, and how to insert a few rows manually. We will use a simple `subjects` table as our example.

In [3]:
with engine.connect() as conn:

    # Drop the table first if it already exists (safe to re-run this cell)
    conn.execute(text('DROP TABLE IF EXISTS subjects'))

    # Create a simple table with three columns
    conn.execute(text('''
        CREATE TABLE subjects (
            subject_id   INT AUTO_INCREMENT PRIMARY KEY,
            subject_name VARCHAR(100) NOT NULL,
            department   VARCHAR(100)
        )
    '''))

    # Insert three dummy rows into the table
    conn.execute(text('''
        INSERT INTO subjects (subject_name, department) VALUES
            ('Mathematics',      'Science & Maths'),
            ('English Language', 'English & Drama'),
            ('Biology',          'Science & Maths')
    '''))

    # Commit saves the changes permanently to the database
    conn.commit()

    print('Table created and dummy rows inserted!')

Table created and dummy rows inserted!


In [4]:
# Verify: read the table back into a DataFrame to confirm the rows are there
df_subjects = pd.read_sql('SELECT * FROM subjects', engine)
df_subjects

,subject_id,subject_name,department
0,1,Mathematics,Science & Maths
1,2,English Language,English & Drama
2,3,Biology,Science & Maths


## Step 4 - Read the CSV files into pandas DataFrames

Now we move to the real data. We have three CSV files - `students.csv`, `teachers.csv`, and `grades.csv`. We read each one into a pandas DataFrame so we can inspect it before loading it into MySQL.

In [5]:
# Read each CSV file into its own DataFrame
# Make sure the CSV files are in the same folder as this notebook
df_students = pd.read_csv('students.csv')
df_teachers = pd.read_csv('teachers.csv')
df_grades   = pd.read_csv('grades.csv')

# Print the shape (rows, columns) of each to confirm they loaded correctly
print(f'students : {df_students.shape[0]} rows, {df_students.shape[1]} columns')
print(f'teachers : {df_teachers.shape[0]} rows, {df_teachers.shape[1]} columns')
print(f'grades   : {df_grades.shape[0]} rows, {df_grades.shape[1]} columns')

students : 30 rows, 11 columns
teachers : 14 rows, 8 columns
grades   : 1620 rows, 10 columns


In [6]:
# Preview the first 3 rows of each DataFrame
print('--- Students ---')
display(df_students.head())

print('--- Teachers ---')
display(df_teachers.head())

print('--- Grades ---')
display(df_grades.head())

--- Students ---


,student_id,first_name,last_name,year_group,gender,house,tutor_group,entry_year,pupil_premium,sen_support,form_tutor_id
0,S001,Amelia,Smith,7,M,Attenborough,7A,2023,N,NaN,T03
1,S002,Noah,Johnson,7,F,Churchill,7B,2023,N,NaN,T03
2,S003,Olivia,Williams,7,M,Darwin,7C,2023,N,NaN,T03
3,S004,Liam,Brown,7,F,Newton,7D,2023,N,Dyslexia,T03
4,S005,Isla,Taylor,7,M,Attenborough,7A,2023,Y,NaN,T03


--- Teachers ---


,teacher_id,first_name,last_name,subject,department,years_experience,is_form_tutor,email
0,T01,Sarah,Matthews,Mathematics,STEM,12,Yes,s.matthews@school.ac.uk
1,T02,James,Okoye,Mathematics,STEM,5,No,j.okoye@school.ac.uk
2,T03,Priya,Sharma,English Language,Humanities,8,Yes,p.sharma@school.ac.uk
3,T04,David,Chen,English Literature,Humanities,15,No,d.chen@school.ac.uk
4,T05,Emma,Thornton,Biology,Science,10,Yes,e.thornton@school.ac.uk


--- Grades ---


,record_id,student_id,subject,academic_year,term,percentage,grade,effort,teacher_id,predicted_grade
0,1,S007,Mathematics,2024-25,Autumn,68.0,C,3.0,T01,B
1,2,S007,Mathematics,2024-25,Spring,73.0,B,4.0,T02,B
2,3,S007,Mathematics,2024-25,Summer,80.0,A,1.0,T01,A
3,4,S007,English Language,2024-25,Autumn,63.0,C,5.0,T04,B
4,5,S007,English Language,2024-25,Spring,75.0,B,3.0,T04,A


## Step 5 - Write the DataFrames into MySQL tables

pandas has a built-in method called `to_sql()` that writes an entire DataFrame into a database table in one line. We use `if_exists='replace'` so the table is recreated fresh each time — useful during development. Use `'append'` instead if you want to add rows to an existing table.

In [7]:
# Write the students DataFrame to a MySQL table called 'students'
# if_exists='replace' drops and recreates the table if it already exists
# index=False prevents pandas from writing its row numbers as an extra column
df_students.to_sql('students', con=engine, if_exists='replace', index=False)
print('students table written to MySQL')

# Write the teachers DataFrame to a MySQL table called 'teachers'
df_teachers.to_sql('teachers', con=engine, if_exists='replace', index=False)
print('teachers table written to MySQL')

# Write the grades DataFrame to a MySQL table called 'grades'
# This is the largest table — 1620 rows
df_grades.to_sql('grades', con=engine, if_exists='replace', index=False)
print('grades table written to MySQL')

print('\nAll three tables loaded successfully!')

students table written to MySQL
teachers table written to MySQL
grades table written to MySQL

All three tables loaded successfully!


In [8]:
# Confirm the tables now exist in MySQL by listing all tables in the database
with engine.connect() as conn:
    result = conn.execute(text('SHOW TABLES'))
    tables = [row[0] for row in result]  # extract table names from the result

print('Tables in school_db:', tables)

Tables in school_db: ['grades', 'students', 'subjects', 'teachers']


## Step 6 - Read data back from MySQL into pandas

Now that the data lives in MySQL, we can query it using standard SQL and load the results directly into a pandas DataFrame. This is the most common workflow — use SQL to filter and join data in the database, then bring the result into pandas for analysis or visualisation.

In [9]:
# Simple query: read the entire students table back into a DataFrame
df_from_db = pd.read_sql('SELECT * FROM students', engine)

print(f'Loaded {len(df_from_db)} rows from the students table')
df_from_db.head()

Loaded 30 rows from the students table


,student_id,first_name,last_name,year_group,gender,house,tutor_group,entry_year,pupil_premium,sen_support,form_tutor_id
0,S001,Amelia,Smith,7,M,Attenborough,7A,2023,N,None,T03
1,S002,Noah,Johnson,7,F,Churchill,7B,2023,N,None,T03
2,S003,Olivia,Williams,7,M,Darwin,7C,2023,N,None,T03
3,S004,Liam,Brown,7,F,Newton,7D,2023,N,Dyslexia,T03
4,S005,Isla,Taylor,7,M,Attenborough,7A,2023,Y,None,T03


In [10]:
# Filtered query: only students in Year 10 who receive Pupil Premium
query = '''
    SELECT student_id, first_name, last_name, year_group, pupil_premium
    FROM   students
    WHERE  year_group = 10
    AND    pupil_premium = 'Y'
'''

# The SQL result is returned directly as a pandas DataFrame
df_pp_y10 = pd.read_sql(query, engine)
print(f'{len(df_pp_y10)} Pupil Premium students in Year 10')
df_pp_y10

1 Pupil Premium students in Year 10


,student_id,first_name,last_name,year_group,pupil_premium
0,S020,Theo,White,10,Y


In [13]:
# Join query: combine grades with student names in one SQL statement
# This avoids loading all rows into pandas before filtering
query = '''
    SELECT
        s.first_name,
        s.last_name,
        g.subject,
        g.term,
        g.percentage,
        g.grade
    FROM  grades g
    JOIN  students s ON g.student_id = s.student_id
    WHERE g.subject = 'Mathematics'
    ORDER BY g.percentage DESC
    LIMIT 10
'''

# Top 10 Maths results across all students
df_top_maths = pd.read_sql(query, engine)
print('Top 10 Mathematics results:')
df_top_maths

Top 10 Mathematics results:


,first_name,last_name,subject,term,percentage,grade
0,Chloe,Thompson,Mathematics,Autumn,100.0,A*
1,Evelyn,Patel,Mathematics,Spring,95.0,A*
2,Chloe,Thompson,Mathematics,Spring,93.0,A*
3,Chloe,Thompson,Mathematics,Autumn,93.0,A*
4,Theo,White,Mathematics,Summer,92.0,A*
5,Muhammad,Adams,Mathematics,Summer,92.0,A*
6,Muhammad,Adams,Mathematics,Spring,91.0,A*
7,Theo,White,Mathematics,Summer,91.0,A*
8,Muhammad,Adams,Mathematics,Autumn,91.0,A*
9,Muhammad,Adams,Mathematics,Summer,91.0,A*


In [14]:
# Aggregation query: average percentage per subject across all terms
query = '''
    SELECT
        subject,
        ROUND(AVG(percentage), 1) AS avg_pct,   -- average score per subject
        COUNT(*)                  AS total_records
    FROM  grades
    GROUP BY subject
    ORDER BY avg_pct DESC
'''

df_avg = pd.read_sql(query, engine)
print('Average score by subject:')
df_avg

Average score by subject:


,subject,avg_pct,total_records
0,Chemistry,65.3,162
1,Computing,64.7,162
2,French,64.6,162
3,Art & Design,64.6,162
4,English Language,64.1,162
5,Mathematics,63.8,162
6,History,63.8,162
7,Biology,63.7,162
8,Physics,63.4,162
9,Geography,63.0,162


## Summary

| Step | What we did |
|------|-------------|
| 1 | Installed `pymysql` and `sqlalchemy` |
| 2 | Connected to MySQL using `create_engine()` |
| 3 | Created a table manually with SQL and inserted dummy rows |
| 4 | Read three CSV files into pandas DataFrames with `read_csv()` |
| 5 | Wrote the DataFrames into MySQL with `to_sql()` |
| 6 | Queried the MySQL tables back into pandas with `read_sql()` |

**Key methods to remember:**
- `pd.read_csv('file.csv')` — read a CSV file
- `df.to_sql('table_name', engine, if_exists='replace', index=False)` — write to MySQL
- `pd.read_sql('SELECT ...', engine)` — query MySQL into a DataFrame